# Modul 13: Klassische Bild- und Signalmodelle

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Klassische Bildmodelle, Signalmodelle  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 135 bis 185 Minuten

    ## Überblick

    Sie erzeugen klassische Merkmale aus Bildern und Signalfenstern und trainieren scikit-learn-Pipelines darauf. Leakage-freie Splits, zeitliche Bewertung, Konfusionsmatrix, Fehlbilder und Vorhersageverläufe verbinden beide Datentypen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_13A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_13B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Kleine Bilddaten als Pixelvektoren oder klassische Bildmerkmale vorbereiten.
- Klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage trainieren.
- Bildmodelle mit Konfusionsmatrix und Fehlklassifikationen bewerten.
- Fenster und Statistik- sowie Frequenzmerkmale aus Signalen erzeugen.
- scikit-learn-Pipelines für Signal-Klassifikation oder Regression trainieren.
- Signalmodelle zeitlich korrekt bewerten und mit Baselines vergleichen.

    ## Bewertete Fähigkeiten

    - Digits, Pixelvektoren, Histogramme, Kanten und HOG
- PCA in Pipelines, LogisticRegression und SVC
- Fensterlabels, Statistik- und FFT-Merkmale
- zeitlicher Split, TimeSeriesSplit und Vorhersagevisualisierung

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skimage.feature import hog

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

digits_13 = load_digits()
images_13 = digits_13.images.astype(np.float32)
labels_13 = digits_13.target

# Kontinuierliches Signal mit wechselnden Zuständen für eine zeitliche Aufgabe.
sampling_rate_13 = 50
segment_length_13 = 100
state_frequencies_13 = [3, 3, 8, 8, 3, 8, 3, 8, 8, 3, 3, 8]
signal_parts_13 = []
state_labels_13 = []
for state_index, frequency in enumerate(state_frequencies_13):
    local_time = np.arange(segment_length_13) / sampling_rate_13
    part = (
        np.sin(2 * np.pi * frequency * local_time)
        + 0.25 * np.sin(2 * np.pi * (frequency + 2) * local_time)
        + rng.normal(0, 0.18, segment_length_13)
    )
    signal_parts_13.append(part)
    state_labels_13.extend([0 if frequency == 3 else 1] * segment_length_13)
continuous_signal_13 = np.concatenate(signal_parts_13)
point_labels_13 = np.array(state_labels_13)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Pixel-, Histogramm-, Kanten- und HOG-Merkmale erzeugen

    Erstellen Sie für jedes Digits-Bild vier Merkmalsdarstellungen:

1. rohe 64 Pixelwerte,
2. Intensitätshistogramm mit acht Bins,
3. einfache Kantenmerkmale aus horizontalen und vertikalen Differenzen,
4. HOG-Merkmale mit für 8x8-Bilder geeigneten kleinen Zellen.

Geben Sie Formen und erste Merkmalszeilen aus. Visualisieren Sie ein Originalbild und seine Gradientenstärke in getrennten Abbildungen.

> **Hinweis:** Prüfen Sie, ob jedes Bild genau eine Merkmalszeile erzeugt.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Pixel-, Histogramm-, Kanten- und HOG-Merkmale erzeugen
#
# Ziel dieser Codezelle:
# Erstellen Sie für jedes Digits-Bild vier Merkmalsdarstellungen: 1. rohe 64
# Pixelwerte, 2. Intensitätshistogramm mit acht Bins, 3. einfache Kantenmerkmale aus
# horizontalen und vertikalen Differenzen, 4. HOG-Merkmale mi...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Rohe Pixel werden pro Bild zu einem Vektor flachgelegt.
raw_pixel_features = images_13.reshape(len(images_13), -1)

# Histogramme zählen Intensitäten unabhängig von ihrer Position.
histogram_features = np.vstack(
    [
        np.histogram(image, bins=8, range=(0, 16), density=True)[0]
        for image in images_13
    ]
)

edge_feature_rows = []
gradient_magnitude_images = []
for image in images_13:
    gradient_y, gradient_x = np.gradient(image)
    magnitude = np.sqrt(gradient_x**2 + gradient_y**2)
    gradient_magnitude_images.append(magnitude)

    # Mittelwert und Standardabweichung getrennt nach x, y und Betrag
    # liefern eine kleine klassische Kantenbeschreibung.
    edge_feature_rows.append(
        [
            gradient_x.mean(),
            gradient_x.std(),
            gradient_y.mean(),
            gradient_y.std(),
            magnitude.mean(),
            magnitude.std(),
            magnitude.max(),
        ]
    )
edge_features = np.asarray(edge_feature_rows, dtype=np.float32)
gradient_magnitude_images = np.asarray(gradient_magnitude_images)

# Kleine 4x4-Zellen passen zur 8x8-Auflösung. cells_per_block=(1,1)
# vermeidet eine zu große Blockstruktur.
hog_features = np.vstack(
    [
        hog(
            image,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(1, 1),
            feature_vector=True,
        )
        for image in images_13
    ]
)

print("Rohpixel:", raw_pixel_features.shape)
print("Histogramm:", histogram_features.shape)
print("Kantenmerkmale:", edge_features.shape)
print("HOG:", hog_features.shape)
print("Erste HOG-Zeile:", np.round(hog_features[0], 3))

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(images_13[0], cmap="gray")
ax.set_title(f"Original, Label {labels_13[0]}")
ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(gradient_magnitude_images[0], cmap="gray")
ax.set_title("Gradientenstärke")
ax.axis("off")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Rohpixel erhalten Position und Intensität, reagieren aber direkt auf Verschiebungen. Histogramme verlieren räumliche Anordnung. Die kleinen Kantenstatistiken fassen nur globale Gradienten zusammen. HOG bewahrt lokale Orientierungsinformationen und ist oft robuster gegenüber Helligkeitsänderungen. Jede Darstellung besitzt andere Informationsverluste und Modellannahmen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Bildklassifikatoren leakage-frei vergleichen

    Erzeugen Sie einmalig stratifizierte Train/Test-Indizes und wenden Sie dieselben Indizes auf alle Merkmalsmatrizen an.

Vergleichen Sie:

- Dummy-Baseline,
- skalierte logistische Regression auf Rohpixeln,
- skalierte logistische Regression auf HOG,
- PCA(20) + skalierte logistische Regression auf Rohpixeln,
- skalierte SVC auf HOG.

Berechnen Sie Testgenauigkeit und speichern Sie Vorhersagen.

> **Hinweis:** Feature-Extraktion ohne gelernte Parameter darf vor dem Split erfolgen; gelernte PCA dagegen nicht.

In [ ]:
all_indices = np.arange(len(labels_13))

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Bildklassifikatoren leakage-frei vergleichen
#
# Ziel dieser Codezelle:
# Erzeugen Sie einmalig stratifizierte Train/Test-Indizes und wenden Sie dieselben
# Indizes auf alle Merkmalsmatrizen an. Vergleichen Sie: - Dummy-Baseline, -
# skalierte logistische Regression auf Rohpixeln, - skalierte l...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

all_indices = np.arange(len(labels_13))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=labels_13,
)
y_train = labels_13[train_indices]
y_test = labels_13[test_indices]

image_models = {
    "Dummy Rohpixel": (raw_pixel_features, DummyClassifier(strategy="most_frequent")),
    "LogReg Rohpixel": (
        raw_pixel_features,
        Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)),
            ]
        ),
    ),
    "LogReg HOG": (
        hog_features,
        Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)),
            ]
        ),
    ),
    "PCA + LogReg": (
        raw_pixel_features,
        Pipeline(
            [
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=20, random_state=RANDOM_SEED)),
                ("model", LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)),
            ]
        ),
    ),
    "SVC HOG": (
        hog_features,
        Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", SVC(C=5.0, gamma="scale")),
            ]
        ),
    ),
}

image_result_rows = []
image_predictions = {}
fitted_image_models = {}

for name, (features, model) in image_models.items():
    X_train = features[train_indices]
    X_test = features[test_indices]
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    fitted_image_models[name] = model
    image_predictions[name] = predictions
    image_result_rows.append(
        {
            "model": name,
            "feature_count": features.shape[1],
            "accuracy": accuracy_score(y_test, predictions),
        }
    )

image_model_comparison = (
    pd.DataFrame(image_result_rows)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)
print(image_model_comparison.round(3).to_string(index=False))

### Reflexion zu Aufgabe 2

Der gemeinsame Indexsplit verhindert, dass unterschiedliche Modelle auf unterschiedlich schwierigen Testbildern verglichen werden. PCA wird innerhalb der Pipeline ausschließlich auf Trainingsdaten gefittet. HOG kann mit deutlich weniger Merkmalen gute Leistung liefern, während Rohpixel mehr Detail enthalten. Die beste Darstellung hängt von Datenvariation und Modellfamilie ab.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Konfusionsmatrix, PCA und Fehlklassifikationen analysieren

    Wählen Sie das beste Bildmodell aus Aufgabe 2.

1. Erstellen Sie seine Konfusionsmatrix als DataFrame.
2. Identifizieren Sie die fünf häufigsten Verwechslungspaare außerhalb der Diagonale.
3. Zeigen Sie bis zu zwölf falsch klassifizierte Bilder in einer gemeinsamen Matplotlib-Abbildung mit wahrem und vorhergesagtem Label.
4. Falls das PCA-Modell trainiert wurde, geben Sie kumulierte erklärte Varianz der 20 Komponenten aus.

> **Hinweis:** Analysieren Sie sowohl häufige Verwechslungspaare als auch konkrete Bilder.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Konfusionsmatrix, PCA und Fehlklassifikationen analysieren
#
# Ziel dieser Codezelle:
# Wählen Sie das beste Bildmodell aus Aufgabe 2. 1. Erstellen Sie seine
# Konfusionsmatrix als DataFrame. 2. Identifizieren Sie die fünf häufigsten
# Verwechslungspaare außerhalb der Diagonale. 3. Zeigen Sie bis zu zwölf fa...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

best_image_name = image_model_comparison.loc[0, "model"]
best_image_predictions = image_predictions[best_image_name]

matrix = confusion_matrix(y_test, best_image_predictions, labels=np.arange(10))
confusion_df = pd.DataFrame(
    matrix,
    index=[f"true_{i}" for i in range(10)],
    columns=[f"pred_{i}" for i in range(10)],
)

# Die Diagonale wird auf null gesetzt, damit nur Verwechslungen gerankt werden.
off_diagonal = matrix.copy()
np.fill_diagonal(off_diagonal, 0)
flat_order = np.argsort(off_diagonal.ravel())[::-1]
confusion_pairs = []
for flat_index in flat_order:
    true_label, predicted_label = np.unravel_index(flat_index, off_diagonal.shape)
    count = off_diagonal[true_label, predicted_label]
    if count == 0:
        break
    confusion_pairs.append(
        {
            "true_label": true_label,
            "predicted_label": predicted_label,
            "count": int(count),
        }
    )
    if len(confusion_pairs) == 5:
        break

wrong_positions = np.flatnonzero(best_image_predictions != y_test)[:12]
if len(wrong_positions) > 0:
    fig, axes = plt.subplots(3, 4, figsize=(10, 7))
    axes = axes.ravel()
    for ax in axes:
        ax.axis("off")
    for ax, wrong_position in zip(axes, wrong_positions):
        original_index = test_indices[wrong_position]
        ax.imshow(images_13[original_index], cmap="gray")
        ax.set_title(
            f"wahr {y_test[wrong_position]} | vorh. {best_image_predictions[wrong_position]}"
        )
        ax.axis("off")
    plt.tight_layout()
    plt.show()

pca_pipeline = fitted_image_models["PCA + LogReg"]
cumulative_variance = pca_pipeline.named_steps["pca"].explained_variance_ratio_.cumsum()

print("Bestes Modell:", best_image_name)
print("\nKonfusionsmatrix:")
print(confusion_df)
print("\nHäufigste Verwechslungen:")
print(pd.DataFrame(confusion_pairs).to_string(index=False))
print("\nKumulierte PCA-Varianz nach 20 Komponenten:", round(cumulative_variance[-1], 3))

### Reflexion zu Aufgabe 3

Die Konfusionsmatrix zeigt, welche Ziffern systematisch ähnlich erscheinen. Einzelne Fehlbilder helfen zu erkennen, ob Handschrift, geringe Kontraste oder ungewöhnliche Formen beteiligt sind. PCA-Varianz beschreibt erhaltene Pixelvariation, nicht direkt erhaltene Klassifikationsinformation. Ein niedriger Fehleranteil sollte nicht darüber hinwegtäuschen, dass bestimmte Klassen schwerer sein können.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Signalfenster und zeitlich geordnete Merkmalstabelle erzeugen

    Zerlegen Sie `continuous_signal_13` in nicht überlappende Fenster der Länge 100. Für jedes Fenster:

- übernehmen Sie das Mehrheitslabel der Punktlabels,
- berechnen Sie Mittelwert, Standardabweichung, RMS, Peak-to-Peak, maximale Änderung,
- berechnen Sie dominante Frequenz und Spektralenergie in 0 bis 5 Hz sowie 5 bis 15 Hz,
- speichern Sie Startzeit und Endzeit.

Erstellen Sie eine nach Zeit sortierte Merkmalstabelle und prüfen Sie die Klassenverteilung.

> **Hinweis:** Die Zielspalte gehört nicht in die Merkmalsmatrix.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Signalfenster und zeitlich geordnete Merkmalstabelle erzeugen
#
# Ziel dieser Codezelle:
# Zerlegen Sie continuoussignal13 in nicht überlappende Fenster der Länge 100. Für
# jedes Fenster: - übernehmen Sie das Mehrheitslabel der Punktlabels, - berechnen
# Sie Mittelwert, Standardabweichung, RMS, Peak-to-Peak, m...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

window_size = 100
signal_rows = []

for start in range(0, len(continuous_signal_13) - window_size + 1, window_size):
    end = start + window_size
    window = continuous_signal_13[start:end]
    label_window = point_labels_13[start:end]

    # Mehrheitslabel ist hier eindeutig, weil die Fenster mit den
    # erzeugten Zustandssegmenten übereinstimmen.
    label = int(np.bincount(label_window).argmax())

    centered = window - window.mean()
    spectrum = np.abs(np.fft.rfft(centered))
    frequencies = np.fft.rfftfreq(window_size, d=1 / sampling_rate_13)
    power = spectrum**2
    spectrum[0] = 0.0

    low_band = (frequencies >= 0) & (frequencies < 5)
    high_band = (frequencies >= 5) & (frequencies <= 15)

    signal_rows.append(
        {
            "start_index": start,
            "end_index": end,
            "start_time_s": start / sampling_rate_13,
            "end_time_s": end / sampling_rate_13,
            "mean": float(window.mean()),
            "std": float(window.std(ddof=0)),
            "rms": float(np.sqrt(np.mean(window**2))),
            "peak_to_peak": float(np.ptp(window)),
            "max_abs_change": float(np.max(np.abs(np.diff(window)))),
            "dominant_frequency_hz": float(frequencies[np.argmax(spectrum)]),
            "energy_0_5_hz": float(power[low_band].sum()),
            "energy_5_15_hz": float(power[high_band].sum()),
            "label": label,
        }
    )

signal_feature_table_13 = pd.DataFrame(signal_rows).sort_values(
    "start_index"
).reset_index(drop=True)

print(signal_feature_table_13.round(3).to_string(index=False))
print("\nKlassenverteilung:")
print(signal_feature_table_13["label"].value_counts().sort_index().to_string())

### Reflexion zu Aufgabe 4

Jede Zeile repräsentiert ein vollständiges Zeitfenster. Dominante Frequenz und Bandenergie sollten die Zustände mit 3 Hz und 8 Hz gut unterscheiden. Da die Fenster zeitlich geordnet und aus einem kontinuierlichen Signal stammen, dürfen sie nicht unkritisch zufällig gemischt werden. Überlappende Fenster würden die Abhängigkeit noch verstärken.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Signalpipeline mit zeitlicher Bewertung

    Verwenden Sie die Merkmalstabelle aus Aufgabe 4.

1. Nehmen Sie die ersten acht Fenster als Training und die letzten vier als Test.
2. Vergleichen Sie Mehrheitsklassen-Dummy, skalierte logistische Regression und skalierte SVC.
3. Führen Sie zusätzlich `TimeSeriesSplit(n_splits=4)` nur auf dem Trainingsabschnitt durch.
4. Erstellen Sie eine Zeitdarstellung aus wahrem und vorhergesagtem Testzustand.
5. Dokumentieren Sie, warum eine zufällige Kreuzvalidierung hier irreführend sein könnte.

> **Hinweis:** Zeitliche Reihenfolge gehört zur Datenentstehung und damit zur Bewertungslogik.

In [ ]:
# Führen Sie Aufgabe 4 zuerst aus.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Signalpipeline mit zeitlicher Bewertung
#
# Ziel dieser Codezelle:
# Verwenden Sie die Merkmalstabelle aus Aufgabe 4. 1. Nehmen Sie die ersten acht
# Fenster als Training und die letzten vier als Test. 2. Vergleichen Sie
# Mehrheitsklassen-Dummy, skalierte logistische Regression und skalie...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

feature_columns = [
    "mean",
    "std",
    "rms",
    "peak_to_peak",
    "max_abs_change",
    "dominant_frequency_hz",
    "energy_0_5_hz",
    "energy_5_15_hz",
]
X_signal = signal_feature_table_13[feature_columns]
y_signal = signal_feature_table_13["label"]

split_position = 8
X_signal_train = X_signal.iloc[:split_position]
X_signal_test = X_signal.iloc[split_position:]
y_signal_train = y_signal.iloc[:split_position]
y_signal_test = y_signal.iloc[split_position:]

signal_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "LogReg": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]
    ),
    "SVC": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(C=2.0, gamma="scale")),
        ]
    ),
}

signal_rows = []
signal_predictions = {}
for name, model in signal_models.items():
    model.fit(X_signal_train, y_signal_train)
    predictions = model.predict(X_signal_test)
    signal_predictions[name] = predictions
    signal_rows.append(
        {"model": name, "test_accuracy": accuracy_score(y_signal_test, predictions)}
    )

# Zeitliche CV wächst von früheren Trainingsfenstern zu späteren
# Validierungsfenstern. Mit nur acht Fenstern sind Ergebnisse unsicher.
time_cv = TimeSeriesSplit(n_splits=4)
cv_scores = cross_validate(
    signal_models["LogReg"],
    X_signal_train,
    y_signal_train,
    cv=time_cv,
    scoring="accuracy",
)

signal_comparison = pd.DataFrame(signal_rows).sort_values(
    "test_accuracy",
    ascending=False,
)
best_signal_name = signal_comparison.iloc[0]["model"]
best_signal_predictions = signal_predictions[best_signal_name]

test_times = signal_feature_table_13.iloc[split_position:]["start_time_s"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(test_times, y_signal_test.to_numpy(), where="post", label="wahr")
ax.step(test_times, best_signal_predictions, where="post", label="vorhergesagt")
ax.set_title(f"Testzustände: {best_signal_name}")
ax.set_xlabel("Fensterstart in Sekunden")
ax.set_ylabel("Zustand")
ax.set_yticks([0, 1])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(signal_comparison.round(3).to_string(index=False))
print("TimeSeriesSplit-Genauigkeiten:", np.round(cv_scores["test_score"], 3))

### Reflexion zu Aufgabe 5

Eine zufällige Aufteilung könnte spätere Zustandsmuster in das Training bringen und bei überlappenden Fenstern nahezu identische Signalabschnitte auf beide Seiten verteilen. Der zeitliche Split simuliert die Vorhersage zukünftiger Fenster. Wegen der sehr kleinen Zahl von Segmenten sind einzelne Genauigkeiten nur didaktische Hinweise und keine belastbare Leistungsangabe.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.